In [45]:
import os
import pickle
import numpy as np
import pandas as pd
import anndata as ad
from pycisTopic.cistopic_class import create_cistopic_object
from pycisTopic.lda_models import run_cgs_models, evaluate_models
from pycisTopic.topic_binarization import binarize_topics
from pycisTopic.diff_features import impute_accessibility, normalize_scores, find_highly_variable_features, find_diff_features
from pycisTopic.utils import region_names_to_coordinates

In [5]:
path_data = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/scratch/2025_11_30/peak_matrix.h5ad"
cluster_key = "cell_type"
organism = "human"
path_out = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/scratch/2025_11_30"

In [ ]:
# Read atac adata
adata = ad.read_h5ad(path_data)
print(f"adata.obs.index.duplicated().sum() before: {adata.obs.index.duplicated().sum()}")
adata.obs.index = adata.obs["cell_type"].astype(str) + "#" + adata.obs.index.astype(str)
print(f"adata.obs.index.duplicated().sum() after: {adata.obs.index.duplicated().sum()}")
adata

/cellar/users/aklie/opt/miniconda3/envs/test_scenicplus_dev/lib/python3.11/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 103954 × 678574
    obs: 'cell_type'

In [ ]:
# Blacklists
if organism == 'human':
    path_blacklist = '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/ref/blacklist.bed.gz'
elif organism == 'mouse':
    path_blacklist = 'resources/blacklists/mouse.bed'

In [18]:
# Create cisTopic object
cistopic_obj = create_cistopic_object(
    fragment_matrix=adata.to_df().T,
    cell_names=adata.obs.index.values,
    region_names=adata.var.index.values,
    path_to_blacklist=path_blacklist,
    split_pattern="_",
    tag_cells=False,
)

2025-12-04 22:47:44,303 cisTopic     INFO     Converting fragment matrix to sparse matrix
2025-12-04 22:56:34,805 cisTopic     INFO     Removing blacklisted regions
2025-12-04 22:56:37,202 cisTopic     INFO     Creating CistopicObject
2025-12-04 22:56:48,616 cisTopic     INFO     Done!


In [52]:
# Add cell metadata
cistopic_obj.cell_data = cistopic_obj.cell_data.merge(adata.obs, left_index=True, right_index=True)
cistopic_obj.cell_names = cistopic_obj.cell_data.index.values
cistopic_obj.cell_data

,cisTopic_nr_frag,cisTopic_log_nr_frag,cisTopic_nr_acc,cisTopic_log_nr_acc,sample_id,cell_type
D7_GT_POLG2#AAACCGCGTCATCATC-1,17842,4.251444,15053,4.177623,cisTopic,D7_GT_POLG2
D7_GT_POLG2#AAACGCGCAGGCCATT-1,8657,3.937367,7770,3.890421,cisTopic,D7_GT_POLG2
D7_GT_POLG2#AAACGGATCCCGAAGC-1,12453,4.095274,11025,4.042379,cisTopic,D7_GT_POLG2
D7_GT_POLG2#AAAGCACCACATTGCA-1,1801,3.255514,1706,3.231979,cisTopic,D7_GT_POLG2
D7_GT_POLG2#AAAGCCCGTTAGGCGT-1,14941,4.17438,12102,4.082857,cisTopic,D7_GT_POLG2
...,...,...,...,...,...,...
D4_DE_ERBB4+#TTTGTGTTCATTACAG-1,4296,3.633064,4133,3.616265,cisTopic,D4_DE_ERBB4+
D4_DE_ERBB4+#TTTGTGTTCTGGTCCT-1,7365,3.867173,6995,3.844788,cisTopic,D4_DE_ERBB4+
D4_DE_ERBB4+#TTTGTTGGTAATTAGC-1,419,2.622214,402,2.604226,cisTopic,D4_DE_ERBB4+
D4_DE_ERBB4+#TTTGTTGGTCCTAGTT-1,371,2.569374,366,2.563481,cisTopic,D4_DE_ERBB4+


In [ ]:
np.all(cistopic_obj.cell_names == adata.obs.index.values), np.all(cistopic_obj.cell_names == cistopic_obj.cell_data.index.values)

True

In [41]:
adata.write(os.path.join(path_out, "peak_matrix_updated.h5ad"))

In [55]:
pickle.dump(cistopic_obj, open(os.path.join(path_out, "cistopic_obj.pkl"), "wb"))

In [56]:
# Test load
pickle.load(open(os.path.join(path_out, "cistopic_obj.pkl"), "rb"))

In [57]:
np.all(cistopic_obj.cell_names == adata.obs.index.values), np.all(cistopic_obj.cell_names == cistopic_obj.cell_data.index.values)

(True, True)

In [58]:
cistopic_obj.binary_matrix.shape, cistopic_obj.fragment_matrix.shape

((678198, 103954), (678198, 103954))

In [70]:
cistopic_obj.cell_data.index.duplicated().sum()

0

In [63]:
adata.obs.index.duplicated().sum()

0

In [69]:
np.unique(np.unique(cistopic_obj.cell_names, return_counts=True)[1])

array([1])

# DONE

---